# Point deduplicator for virma and lipas points

***
***NOTE*** <br>
Experimental code, not suitable for every dataset at this stage.
***

The objective of this deduplicator was to match points together between two spatial applications which both describe different outdoor sports and hiking places. We wanted to match the points from both applications that represented the same location in the real world. 

The deduplication process was done by first performing a K nearest neighbor search in QGIS based on the location of the points. From this search, the 3 nearest Lipas points within 200 meters were selected for each Virma point. This gave us a csv dataset from which we could determine whether any of the three closest points represented the same destination as the target Virma point.

From the three NN the nearest neighbor was decided based on the similarity of the points. Meaning that, in our dataset our points have a lot of metadata in addition to the location, such as name, category of the point, city etc. The points were considered the same if they had the same category or the exact same name since e.g., a beach could be "Uimaranta" in Lipas and "Uimapaikka" in Virma. For this we had a category mapping between our two applications the mapping.csv, which during this development was not quite complete yet but good enough. Borderline cases were decided by human. 

For those points that had so called "mismatched" categories, since they could be different in each application, we performed a similarity calculation based on the names, so that it could be easier to notice such cases where the names between the points were not exactly the same and the categories didn't match, but the points still represented the same destination. 

This application is all in all very simple and with this specific dataset a lot of manual work had to be put in so that all individual cases were spotted. In high level this works but some uniqueness may be overlooked.

Original dataset or the code output is not provided because of privacy reasons and the results are in virma_lipas_points_real.csv and the points that could be matched are in readable form in virma_lipas_points_matched.
***

In [85]:
import pandas as pd
pd.reset_option("display.max_rows")

category_mapping_df = pd.read_csv("mappings.csv")

mapping_dict = dict(zip(category_mapping_df['class2_fi'], category_mapping_df['type_name']))

In [86]:
category_mapping_df.head()

,class2_fi,type_code,type_name
0,Ankkuripaikka,NaN,NaN
1,Vieraslaituri,203.0,Veneilyn palvelupaikka
2,Kulttuuriulkoilureitti,NaN,NaN
3,Majoituspalvelu,NaN,NaN
4,Virkistykseen liittyvä erityiskohde,NaN,NaN


In [87]:
mapping_dict

{'Ankkuripaikka': nan,
 'Vieraslaituri': 'Veneilyn palvelupaikka',
 'Kulttuuriulkoilureitti': nan,
 'Majoituspalvelu': nan,
 'Virkistykseen liittyvä erityiskohde': nan,
 'Yhteysaluslaituri': nan,
 'Päivälaavu tai -kota,301,"Laavu, kota tai kammi"': nan,
 'Retkeilyä palveleva parkkipaikka': nan,
 'Rantautumispaikka': 'Rantautumispaikka',
 'Levähdyspaikka': nan,
 'Luonnonmuistomerkki tai näköalapaikka': nan,
 'Matkailureitti': nan,
 'Tupa': 'Tupa',
 'Veneenlaskupaikka': 'Veneilyn palvelupaikka',
 'Sauna': nan,
 'Käyntisatama': 'Veneilyn palvelupaikka',
 'Opastuskeskus': 'Opastuspiste',
 'Talviuimapaikka': 'Talviuintipaikka',
 'Varaussauna': nan,
 'Ruokapalvelu': nan,
 'Leirikeskus': nan,
 'Suojasatama': nan,
 'Melontareitti': 'Melontareitti',
 'Tilavuokraus- tai kokouspalvelu': nan,
 'Hätäsatama': nan,
 'Virkistysreitin lähtöpiste': 'Opastuspiste',
 'Tulipaikka': 'Ruoanlaitto-/tulentekopaikka',
 'Opaspalvelu': 'Opastuspiste',
 'Yleisö-wc tai -puucee': nan,
 'Latu': 'Koirahiihtolatu',
 'V

In [88]:
df = pd.read_csv("../../Data/really_3_NN_virma_lipas.csv")
df

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y
0,74.0,98.0,74,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Lappdalin uimapaikka,...,NaN,99584.0,Uimapaikka,Lappdalin uimapaikka,1.0,11.104503,263509.242614,6.685405e+06,263518.7850,6.685411e+06
1,NaN,NaN,305485,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,"Parmaharjun luontopolku, parkkipaikka",...,NaN,77998.0,Kuntosali,Parmaharjun kuntosali,1.0,50.427351,255311.894240,6.718301e+06,255261.5262,6.718298e+06
2,754.0,781.0,754,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Friskalanlahden lintutorni,...,NaN,523931.0,Luontotorni,Friskalanlahden lintutorni,1.0,3.734996,238165.493442,6.704733e+06,238162.8179,6.704730e+06
3,863.0,890.0,863,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Retkeilyä palveleva parkkipaikka,Parkeringsplats för vandring,Parking lot for hikers,Mikkossuon pysäköintialue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,864.0,891.0,864,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Endalin laavu,...,NaN,74376.0,"Laavu, kota tai kammi",Endalin laavu,1.0,0.939190,276735.172212,6.683015e+06,276734.2335,6.683015e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1657,NaN,NaN,302267,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Ryyppykallion Kotalaavu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1658,NaN,NaN,302268,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Taivassalon Helsingin uimaranta,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1659,NaN,NaN,302219,Virkistyskohde,rekreationsobjekt,recreational attraction,Tulipaikka,eldplats,campfire site,Kyhkärännokka tulentekopaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1660,1115.0,1142.0,1115,Virkistyskohde,rekreationsobjekt,recreational attraction,Yöpymislaavu tai -kota,övernattningsvindskydd eller -kåta,lean-to or lap pole tent for overnighting,Iso-Valkeen pohjoisrannan laavu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [89]:
df[df["name_fi"] == "Rivieran frisbeegolfrata"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y
1607,NaN,NaN,302143,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Rivieran frisbeegolfrata,...,NaN,616943.0,Beachvolley-/rantalentopallokenttä,Isonkivenrannan beachvolleykenttä 2,1.0,49.032451,233120.5,6722306.5,233087.725,6.722343e+06
1608,NaN,NaN,302143,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Rivieran frisbeegolfrata,...,NaN,609544.0,Beachvolley-/rantalentopallokenttä,Isonkivenrannan beachvolleykenttä 1,2.0,53.130294,233120.5,6722306.5,233082.250,6.722343e+06
1609,NaN,NaN,302143,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Rivieran frisbeegolfrata,...,NaN,100389.0,Uimaranta,Isonkiven uimapaikka,3.0,60.438502,233120.5,6722306.5,233096.000,6.722251e+06


In [90]:
df.head(10)

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y
0,74.0,98.0,74,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Lappdalin uimapaikka,...,NaN,99584.0,Uimapaikka,Lappdalin uimapaikka,1.0,11.104503,263509.242614,6.685405e+06,263518.7850,6.685411e+06
1,NaN,NaN,305485,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,"Parmaharjun luontopolku, parkkipaikka",...,NaN,77998.0,Kuntosali,Parmaharjun kuntosali,1.0,50.427351,255311.894240,6.718301e+06,255261.5262,6.718298e+06
2,754.0,781.0,754,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Friskalanlahden lintutorni,...,NaN,523931.0,Luontotorni,Friskalanlahden lintutorni,1.0,3.734996,238165.493442,6.704733e+06,238162.8179,6.704730e+06
3,863.0,890.0,863,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Retkeilyä palveleva parkkipaikka,Parkeringsplats för vandring,Parking lot for hikers,Mikkossuon pysäköintialue,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,864.0,891.0,864,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Endalin laavu,...,NaN,74376.0,"Laavu, kota tai kammi",Endalin laavu,1.0,0.939190,276735.172212,6.683015e+06,276734.2335,6.683015e+06
5,865.0,892.0,865,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yleisö-wc tai -puucee,Offentlig toalett eller uthus,Public lavatory or outhouse,Endalin puucee,...,NaN,74376.0,"Laavu, kota tai kammi",Endalin laavu,1.0,28.096019,276706.174312,6.683016e+06,276734.2335,6.683015e+06
6,73.0,97.0,73,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Krottilanlahden lintutorni,...,NaN,523929.0,Luontotorni,Krottilanlahden lintutorni,1.0,1.722763,231987.622625,6.708416e+06,231989.3179,6.708416e+06
7,66.0,90.0,66,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Hasselbackenin uimapaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,75.0,99.0,75,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Misskärrin uimapaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,76.0,100.0,76,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Leirikeskus,Lägercenter,Summer camp centre,Merisalon leirimaja,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [91]:
df[df["class2_fi"] == "Yöpymislaavu tai -kota"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y
4,864.0,891.0,864,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Endalin laavu,...,NaN,74376.0,"Laavu, kota tai kammi",Endalin laavu,1.0,0.939190,276735.172212,6.683015e+06,276734.2335,6.683015e+06
40,919.0,946.0,919,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Isoholman laavu (Matildanjärvi),...,NaN,73662.0,"Laavu, kota tai kammi","Isoholman laavu, Matildanjärvi",1.0,0.000087,274541.807697,6.682176e+06,274541.8077,6.682176e+06
112,82.0,106.0,82,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Isoholman laavu (meri),...,NaN,72977.0,"Laavu, kota tai kammi","Isoholman laavu, Teijonselkä",1.0,0.000080,274696.331853,6.686093e+06,274696.3318,6.686093e+06
214,169.0,194.0,169,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Karpaloisten laavu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
336,992.0,1019.0,992,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Vicksbäckinlahden laavu,...,NaN,73537.0,"Laavu, kota tai kammi",Vicksbäckinlahden laavu,1.0,0.947552,274804.570213,6.682459e+06,274803.6231,6.682459e+06
489,419.0,445.0,419,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Vajosuon laavu,...,NaN,73221.0,"Laavu, kota tai kammi",Vajosuon laavu,1.0,1.119430,246768.686534,6.736923e+06,246767.6134,6.736924e+06
494,1005.0,1032.0,1005,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Roosinniemen laavu,...,NaN,73630.0,"Laavu, kota tai kammi",Roosinniemen laavu,1.0,0.944709,275420.830113,6.681959e+06,275419.8860,6.681959e+06
602,490.0,517.0,490,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Kullavuoren laavu,...,NaN,510842.0,"Laavu, kota tai kammi",Kullaanvuoren laavu,1.0,42.705143,234947.542059,6.719252e+06,234938.3336,6.719210e+06
631,512.0,539.0,512,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Töykkälän laavu,...,NaN,73715.0,"Laavu, kota tai kammi",Töykkälän laavu,1.0,23.585211,248684.866832,6.739581e+06,248662.3764,6.739588e+06
648,533.0,560.0,533,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Yöpymislaavu tai -kota,Övernattningsvindskydd eller -kåta,Lean-to or lap pole tent for overnighting,Lakjärven laavu 2,...,NaN,74473.0,"Laavu, kota tai kammi",Lakjärven laavu,1.0,1.115715,245802.898436,6.743869e+06,245801.8425,6.743870e+06


In [92]:
duplicated_ids = df['gid'][df['gid'].duplicated(keep=False)]

duplicates_df = df[df['gid'].isin(duplicated_ids)]

duplicates_df

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y
26,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,NaN,529753.0,Beachvolley-/rantalentopallokenttä,Poikon beachvolleykenttä,1.0,3.216919,217581.250000,6.709144e+06,217578.5514,6.709146e+06
27,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,NaN,529754.0,Ruoanlaitto-/tulentekopaikka,Poikon grillikatos,2.0,7.857486,217581.250000,6.709144e+06,217582.5514,6.709137e+06
28,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,NaN,529752.0,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06
29,123.0,148.0,123,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Hiekkahelmen talviuintipaikka,...,NaN,513259.0,Uimaranta,Hiekkahelmi,1.0,2.948727,263034.716684,6.705549e+06,263036.0964,6.705552e+06
30,123.0,148.0,123,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Hiekkahelmen talviuintipaikka,...,NaN,514952.0,Beachvolley-/rantalentopallokenttä,Hiekkahelmen beachvolleykenttä,2.0,31.478038,263034.716684,6.705549e+06,263062.7653,6.705564e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1643,856.0,883.0,856,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Mustajärven uimaranta,...,NaN,516584.0,Talviuintipaikka,Mustajärven talviuintipaikka,1.0,0.000172,215785.008349,6.773355e+06,215785.0082,6.773355e+06
1644,856.0,883.0,856,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Mustajärven uimaranta,...,NaN,101054.0,Uimapaikka,Mustajärven uimapaikka,2.0,1.076273,215785.008349,6.773355e+06,215785.3655,6.773356e+06
1646,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,NaN,99453.0,Uimapaikka,Ristikallion uimaranta,1.0,30.530924,246911.692700,6.710081e+06,246882.5000,6.710072e+06
1647,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,NaN,83057.0,Pallokenttä,Ristikallion koulu pallokenttä,2.0,75.682280,246911.692700,6.710081e+06,246923.8969,6.710007e+06


In [93]:
mapping_dict = dict(zip(category_mapping_df['class2_fi'], category_mapping_df['type_name']))


duplicates_df['mapped_lipas_category'] = duplicates_df['class2_fi'].map(mapping_dict)

duplicates_df['category_match'] = (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['mapped_lipas_category']) | (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['class2_fi']) | (duplicates_df['name_fi'] == duplicates_df['nimi_fi'])

mismatched_rows = duplicates_df[duplicates_df['category_match'] == False]

if not mismatched_rows.empty:
    print("Mismatched Categories:")
    display(mismatched_rows[["name_fi", "nimi_fi", 'class2_fi', 'tyyppi_nimi_fi', 'mapped_lipas_category']])
else:
    print("All categories match!")

Mismatched Categories:


C:\Users\eoylik\AppData\Local\Temp\ipykernel_17336\3321192366.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicates_df['mapped_lipas_category'] = duplicates_df['class2_fi'].map(mapping_dict)
C:\Users\eoylik\AppData\Local\Temp\ipykernel_17336\3321192366.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  duplicates_df['category_match'] = (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['mapped_lipas_category']) | (duplicates_df['tyyppi_nimi_fi'] == duplicates_df['class2_fi']) | (duplicates_df['name

,name_fi,nimi_fi,class2_fi,tyyppi_nimi_fi,mapped_lipas_category
26,Poikon uimapaikka,Poikon beachvolleykenttä,Uimapaikka,Beachvolley-/rantalentopallokenttä,Uimaranta
27,Poikon uimapaikka,Poikon grillikatos,Uimapaikka,Ruoanlaitto-/tulentekopaikka,Uimaranta
29,Hiekkahelmen talviuintipaikka,Hiekkahelmi,Talviuimapaikka,Uimaranta,Talviuintipaikka
30,Hiekkahelmen talviuintipaikka,Hiekkahelmen beachvolleykenttä,Talviuimapaikka,Beachvolley-/rantalentopallokenttä,Talviuintipaikka
31,"Turku, Aurajoki",Elixia Jokivarsi,Vierassatama,Kuntokeskus,Veneilyn palvelupaikka
...,...,...,...,...,...
1635,Laukkarin laavu,Laukkarin uimaranta,Päivälaavu tai -kota,Uimapaikka,NaN
1643,Mustajärven uimaranta,Mustajärven talviuintipaikka,Uimaranta,Talviuintipaikka,Uimaranta
1644,Mustajärven uimaranta,Mustajärven uimapaikka,Uimaranta,Uimapaikka,Uimaranta
1647,Ristikallion uimaranta,Ristikallion koulu pallokenttä,Uimapaikka,Pallokenttä,Uimaranta


In [94]:
mismatched_rows[mismatched_rows["name_fi"] == "Rautilan rantasauna"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
417,340.0,365.0,340,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Sauna,Bastu,Sauna,Rautilan rantasauna,...,Talviuintipaikka,Rautilan talviuintipaikka,1.0,14.206554,209481.588105,6.732321e+06,209495.4000,6.732325e+06,NaN,False
418,340.0,365.0,340,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Sauna,Bastu,Sauna,Rautilan rantasauna,...,Uimapaikka,Rautilan sauna ja uimapaikka,2.0,19.032360,209481.588105,6.732321e+06,209500.5744,6.732320e+06,NaN,False


In [95]:
true_matches = duplicates_df[duplicates_df['category_match'] != False]
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
28,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
34,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
35,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Littoistenjärven lintutorni,2.0,142.964404,246289.500000,6.711695e+06,246327.5000,6.711557e+06,Luontotorni,True
42,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
53,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1550,NaN,NaN,302569,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Eknimen lomakylän uimapaikka,...,Uimapaikka,Ekniemen lomakylän uimaranta,1.0,3.956635,256250.000000,6.684468e+06,256246.1505,6.684467e+06,Uimaranta,True
1565,773.0,800.0,773,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Kyrön maauimala,...,Maauimala/vesipuisto,Kyrön maauimala,2.0,16.319216,268969.583417,6.737190e+06,268963.0806,6.737175e+06,Uimaranta,True
1584,NaN,NaN,302189,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Laukkarin uimaranta,...,Uimapaikka,Laukkarin uimaranta,2.0,22.884561,197766.000000,6.721461e+06,197785.2500,6.721449e+06,Uimaranta,True
1636,NaN,NaN,302188,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Laukkarin laavu,...,"Laavu, kota tai kammi",Laukkarin laavu,2.0,19.975375,197778.000000,6.721434e+06,197780.0000,6.721454e+06,NaN,True


In [96]:
duplicated_ids2 = true_matches['gid'][true_matches['gid'].duplicated(keep=False)]

duplicates_df2 = true_matches[true_matches['gid'].isin(duplicated_ids2)]

duplicates_df2

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
34,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
35,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Littoistenjärven lintutorni,2.0,142.964404,246289.500000,6.711695e+06,246327.5000,6.711557e+06,Luontotorni,True
53,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
54,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Turun Kalikan uimaranta,3.0,103.727194,250980.466581,6.761814e+06,250953.5937,6.761713e+06,Uimaranta,True
558,461.0,487.0,461,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistysreitin lähtöpiste,startpunkt för friluftsleden,starting point for a recreational route,Märynummen retkeilyreitin lähtöpiste,...,Opastuspiste,Märynummen kuntoradan opastuspiste,1.0,4.389262,283053.000000,6.707532e+06,283051.5000,6.707528e+06,Opastuspiste,True
559,461.0,487.0,461,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistysreitin lähtöpiste,startpunkt för friluftsleden,starting point for a recreational route,Märynummen retkeilyreitin lähtöpiste,...,Opastuspiste,Märynummen retkeilyreitin opastuspiste,2.0,6.535261,283053.000000,6.707532e+06,283052.0000,6.707526e+06,Opastuspiste,True
1077,952.0,979.0,952,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Metsälehdon sauna ja Haimion vapaa sauna,...,Talviuintipaikka,Metsälehdon sauna,1.0,0.000305,192771.519658,6.762965e+06,192771.5195,6.762965e+06,Talviuintipaikka,True
1079,952.0,979.0,952,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Metsälehdon sauna ja Haimion vapaa sauna,...,Talviuintipaikka,Haimion talviuintipaikka,3.0,190.519614,192771.519658,6.762965e+06,192814.0958,6.763151e+06,Talviuintipaikka,True
1175,1043.0,1070.0,1043,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Littoistenjärven lintutorni,...,Luontotorni,Littoistenjärven lintutorni,1.0,152.951994,246175.000000,6.711568e+06,246327.5000,6.711557e+06,Luontotorni,True
1176,1043.0,1070.0,1043,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Littoistenjärven lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,2.0,167.226341,246175.000000,6.711568e+06,246290.3762,6.711690e+06,Luontotorni,True


In [97]:
true_matches = true_matches[~((true_matches['name_fi'] == "Järvelän näköala- ja lintutorni") & (true_matches['nimi_fi'] == "Littoistenjärven lintutorni"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Kalikan uimaranta") & (true_matches['nimi_fi'] == "Turun Kalikan uimaranta"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin lähtöpiste") & (true_matches['nimi_fi'] == "Märynummen kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin lähtöpiste") & (true_matches['nimi_fi'] == "Märynummen retkeilyreitin opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Asmandia-laavu") & (true_matches['nimi_fi'] == "Asmandian grillikota"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Metsälehdon sauna ja Haimion vapaa sauna") & (true_matches['nimi_fi'] == "Haimion talviuintipaikka"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Littoistenjärven lintutorni") & (true_matches['nimi_fi'] == "Järvelän näköala- ja lintutorni"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Ellun polun opastuspiste") & (true_matches['nimi_fi'] == "Melassuon kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Märynummen retkeilyreitin opastuspiste") & (true_matches['nimi_fi'] == "Märynummen kuntoradan opastuspiste"))]
true_matches = true_matches[~((true_matches['name_fi'] == "Palaisten uimaranta") & (true_matches['nimi_fi'] == "Palaisten talviuintipaikka"))]

In [98]:
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
28,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
34,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
42,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
53,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
69,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1550,NaN,NaN,302569,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Eknimen lomakylän uimapaikka,...,Uimapaikka,Ekniemen lomakylän uimaranta,1.0,3.956635,256250.000000,6.684468e+06,256246.1505,6.684467e+06,Uimaranta,True
1565,773.0,800.0,773,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Kyrön maauimala,...,Maauimala/vesipuisto,Kyrön maauimala,2.0,16.319216,268969.583417,6.737190e+06,268963.0806,6.737175e+06,Uimaranta,True
1584,NaN,NaN,302189,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Laukkarin uimaranta,...,Uimapaikka,Laukkarin uimaranta,2.0,22.884561,197766.000000,6.721461e+06,197785.2500,6.721449e+06,Uimaranta,True
1636,NaN,NaN,302188,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Laukkarin laavu,...,"Laavu, kota tai kammi",Laukkarin laavu,2.0,19.975375,197778.000000,6.721434e+06,197780.0000,6.721454e+06,NaN,True


In [99]:
df = df.merge(duplicates_df, how='outer', indicator=True)
df = df[df['_merge'] == 'left_only']
df = df.drop(columns=['_merge'])
df

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
14,6.0,10.0,6,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kuusiston uimaranta,...,Uimapaikka,Kuusiston uimaranta,1.0,0.000000,242140.3816,6.703244e+06,242140.3816,6.703244e+06,NaN,NaN
17,8.0,22.0,8,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Pähkinäisen uimaranta,...,Uimapaikka,Pähkinäisen uimaranta,1.0,1.476903,207317.0463,6.699893e+06,207316.4537,6.699895e+06,NaN,NaN
20,10.0,24.0,10,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Saukkorannan talviuintipaikka,...,Talviuintipaikka,Saukkorannan talviuintipaikka,1.0,0.000000,232689.5374,6.709283e+06,232689.5374,6.709283e+06,NaN,NaN
21,11.0,25.0,11,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Pöylinrannan talviuintipaikka,...,Talviuintipaikka,Pöylinrannan talviuintipaikka,1.0,0.000000,248939.6925,6.699411e+06,248939.6925,6.699411e+06,NaN,NaN
22,12.0,29.0,12,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Järvelän lintulava,...,Luontotorni,Järvelän lintulava,1.0,0.000000,246029.0000,6.711872e+06,246029.0000,6.711872e+06,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1648,NaN,NaN,305823,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Soinisten kartano (Neljäntammenkuja),...,Luistelukenttä,Luistelukenttä,1.0,177.802291,227141.0000,6.716957e+06,227224.8018,6.717114e+06,NaN,NaN
1652,NaN,NaN,305825,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Ihminen on muovannut Taimon kylän maisemaa jo ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1657,NaN,NaN,305935,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistysreitin lähtöpiste,startpunkt för friluftsleden,starting point for a recreational route,Tuomiokirkko - pyhiinvaellusreitin aloituspiste,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1658,NaN,NaN,305993,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistykseen liittyvä erityiskohde,speciellt objekt för friluftsliv,special recreational attraction,Yttilän museokoulu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
needs_checking = mismatched_rows[mismatched_rows['mapped_lipas_category'].isna()]
needs_checking

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
60,45.0,69.0,45,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Kalliorannan laavu,...,Pallokenttä,Karunan jalkapallokenttä,1.0,60.259118,255750.142157,6.690257e+06,255753.2500,6.690197e+06,NaN,False
61,45.0,69.0,45,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Kalliorannan laavu,...,Talviuintipaikka,Kalliorannan talviuintipaikka,2.0,88.229005,255750.142157,6.690257e+06,255706.7500,6.690334e+06,NaN,False
68,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,"Laavu, kota tai kammi",Suojalan laavu,1.0,70.880350,264555.572829,6.694371e+06,264626.4475,6.694371e+06,NaN,False
70,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ampumarata,Suojalan ampumarata,3.0,178.281549,264555.572829,6.694371e+06,264582.7094,6.694548e+06,NaN,False
81,62.0,86.0,62,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Virttaan Eräveikkojen maja,...,Kilpahiihtokeskus,Virttaan kilpahiihtokeskus/ Erä-Veikkojen maja,1.0,102.501015,264925.130438,6.766786e+06,265018.9065,6.766745e+06,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1625,NaN,NaN,302171,Virkistyskohde,rekreationsobjekt,recreational attraction,Sauna,bastu,sauna,Särkijärven rantasauna,...,Talviuintipaikka,Särkijärven talviuintipaikka,1.0,14.460376,205451.750000,6.761191e+06,205457.7035,6.761178e+06,NaN,False
1626,NaN,NaN,302171,Virkistyskohde,rekreationsobjekt,recreational attraction,Sauna,bastu,sauna,Särkijärven rantasauna,...,Uimapaikka,Särkijärven uimapaikka,2.0,18.278111,205451.750000,6.761191e+06,205455.3397,6.761173e+06,NaN,False
1629,NaN,NaN,302172,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,Särkijärven parkkipaikka,...,Uimapaikka,Särkijärven uimapaikka,1.0,51.522659,205493.050003,6.761138e+06,205455.3397,6.761173e+06,NaN,False
1630,NaN,NaN,302172,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,Särkijärven parkkipaikka,...,Talviuintipaikka,Särkijärven talviuintipaikka,2.0,53.268298,205493.050003,6.761138e+06,205457.7035,6.761178e+06,NaN,False


In [101]:
mismatched_rows = mismatched_rows.merge(needs_checking, how='outer', indicator=True)
mismatched_rows = mismatched_rows[mismatched_rows['_merge'] == 'left_only']
mismatched_rows = mismatched_rows.drop(columns=['_merge'])
mismatched_rows

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,Liikuntasali,Ristikallion koulun liikuntasali,3.0,122.374159,246911.6927,6.710081e+06,247020.0341,6.710024e+06,Uimaranta,False
1,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,Pallokenttä,Ristikallion koulu pallokenttä,2.0,75.682280,246911.6927,6.710081e+06,246923.8969,6.710007e+06,Uimaranta,False
2,2.0,2.0,2,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Harvaluodon uimapaikka,...,Ulkokuntoilupaikka,Harvaluodon rannan ulkokuntoilulaite,2.0,17.088958,252817.3075,6.701094e+06,252809.0560,6.701109e+06,Uimaranta,False
3,2.0,2.0,2,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Harvaluodon uimapaikka,...,Frisbeegolfrata,Harvaluodon frisbeegolfrata,3.0,111.481132,252817.3075,6.701094e+06,252705.8815,6.701098e+06,Uimaranta,False
4,3.0,4.0,3,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ispoisten uimaranta,...,Talviuintipaikka,Ispoisten talviuintipaikka,2.0,2.756143,239051.3467,6.706951e+06,239054.0957,6.706951e+06,Uimaranta,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,NaN,NaN,305824,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Kaupungin ja maaseudun rajalla (Aurinkotie) op...,...,Liikuntasali,Maijamäen koulun tatamisali,2.0,149.857015,226998.5000,6.714228e+06,226994.9348,6.714378e+06,Opastuspiste,False
437,NaN,NaN,305824,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Kaupungin ja maaseudun rajalla (Aurinkotie) op...,...,Lähiliikuntapaikka,Maijamäen koulun lähiliikuntapaikka,1.0,110.767969,226998.5000,6.714228e+06,227000.4951,6.714339e+06,Opastuspiste,False
438,NaN,NaN,305824,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Kaupungin ja maaseudun rajalla (Aurinkotie) op...,...,Ulkokuntoilupaikka,Maijamäen ulkokuntoilupaikka,3.0,152.815544,226998.5000,6.714228e+06,227005.7451,6.714381e+06,Opastuspiste,False
439,NaN,NaN,305924,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistysreitin lähtöpiste,startpunkt för friluftsleden,starting point for a recreational route,Hevonlinnan reitin lähtöpiste,...,Frisbeegolfrata,Hevonlinnan frisbeegolfrata,1.0,39.882594,283128.0000,6.735533e+06,283089.2945,6.735523e+06,Opastuspiste,False


In [102]:
mismatched_rows[mismatched_rows['class2_fi'] == "Uimaranta"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
4,3.0,4.0,3,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ispoisten uimaranta,...,Talviuintipaikka,Ispoisten talviuintipaikka,2.0,2.756143,239051.346700,6.706951e+06,239054.0957,6.706951e+06,Uimaranta,False
46,114.0,139.0,114,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Särkijärven uimaranta ja rantasauna,...,Uimapaikka,Särkijärven uimapaikka,2.0,4.980859,205454.995202,6.761178e+06,205455.3397,6.761173e+06,Uimaranta,False
47,114.0,139.0,114,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Särkijärven uimaranta ja rantasauna,...,Talviuintipaikka,Särkijärven talviuintipaikka,1.0,2.717606,205454.995202,6.761178e+06,205457.7035,6.761178e+06,Uimaranta,False
59,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Beachvolley-/rantalentopallokenttä,Ekvallan rantalentopallokenttä,2.0,40.568672,236944.259856,6.703199e+06,236904.7598,6.703208e+06,Uimaranta,False
60,134.0,159.0,134,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Klipunkarin uimaranta,...,Talviuintipaikka,Klipunkarin talviuintipaikka,2.0,6.793762,189915.649222,6.726172e+06,189919.0000,6.726178e+06,Uimaranta,False
79,164.0,189.0,164,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Santtion uimaranta,...,Talviuintipaikka,Santtion talviuintipaikka,1.0,4.002288,195265.541266,6.754566e+06,195269.1347,6.754564e+06,Uimaranta,False
90,236.0,261.0,236,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Lehmijärven uimaranta,...,Beachvolley-/rantalentopallokenttä,Lehmijärven beachvolleykenttä 1,3.0,100.430646,283246.970874,6.691272e+06,283147.0291,6.691262e+06,Uimaranta,False
91,236.0,261.0,236,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Lehmijärven uimaranta,...,Lähiliikuntapaikka,Lehmijärven uimarannan leikkipaikka,1.0,89.878522,283246.970874,6.691272e+06,283159.2000,6.691291e+06,Uimaranta,False
96,255.0,280.0,255,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Kokkilan uimaranta,...,Talviuintipaikka,Kokkilan talviuintipaikka,1.0,5.056286,276129.000000,6.693779e+06,276125.2876,6.693782e+06,Uimaranta,False
97,255.0,280.0,255,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Kokkilan uimaranta,...,Beachvolley-/rantalentopallokenttä,Kokkilan uimapaikan beachvolleykenttä 3,3.0,36.840877,276129.000000,6.693779e+06,276119.5291,6.693815e+06,Uimaranta,False


In [103]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Särkijärven uimaranta ja rantasauna") & (mismatched_rows["nimi_fi"] == "Särkijärven uimapaikka")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Liipolan uimaranta") & (mismatched_rows["nimi_fi"] == "Liipolanjärven uimapaikka")]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,NaN,NaN,302189,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Laukkarin uimaranta,...,Uimapaikka,Laukkarin uimaranta,2.0,22.884561,197766.000000,6.721461e+06,197785.2500,6.721449e+06,Uimaranta,True
114,NaN,NaN,302188,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Laukkarin laavu,...,"Laavu, kota tai kammi",Laukkarin laavu,2.0,19.975375,197778.000000,6.721434e+06,197780.0000,6.721454e+06,NaN,True
115,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,Uimapaikka,Ristikallion uimaranta,1.0,30.530924,246911.692700,6.710081e+06,246882.5000,6.710072e+06,Uimaranta,True
116,114.0,139.0,114,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Särkijärven uimaranta ja rantasauna,...,Uimapaikka,Särkijärven uimapaikka,2.0,4.980859,205454.995202,6.761178e+06,205455.3397,6.761173e+06,Uimaranta,False


In [104]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["gid"] == 856) & (mismatched_rows["nimi_fi"] == "Mustajärven uimapaikka")]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,NaN,NaN,302188,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Laukkarin laavu,...,"Laavu, kota tai kammi",Laukkarin laavu,2.0,19.975375,197778.000000,6.721434e+06,197780.0000,6.721454e+06,NaN,True
115,1.0,1.0,1,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Ristikallion uimaranta,...,Uimapaikka,Ristikallion uimaranta,1.0,30.530924,246911.692700,6.710081e+06,246882.5000,6.710072e+06,Uimaranta,True
116,114.0,139.0,114,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,badstrand,beach,Särkijärven uimaranta ja rantasauna,...,Uimapaikka,Särkijärven uimapaikka,2.0,4.980859,205454.995202,6.761178e+06,205455.3397,6.761173e+06,Uimaranta,False
117,744.0,771.0,744,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimaranta,badstrand,beach,Liipolan uimaranta,...,Uimapaikka,Liipolanjärven uimapaikka,2.0,4.986310,288694.000000,6.726237e+06,288689.1488,6.726238e+06,Uimaranta,False


In [105]:
needs_checking[needs_checking["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

array([    53,   1051,     85,     24,     26,    265, 302389, 302568,
       302575, 302188])

In [106]:
mismatched_rows[mismatched_rows["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

array([     1,      2,      3,      4,      5,      7,      9,     19,
           21,     27,     28,     36,     55,     56,     68,     81,
           94,    101,    106,    107,    109,    111,    113,    114,
          115,    117,    121,    122,    126,    131,    133,    134,
          135,    136,    137,    144,    145,    149,    151,    152,
          164,    236,    247,    255,    292,    314,    339,    348,
          406,    453,    454,    455,    456,    457,    462,    463,
          464,    465,    466,    467,    500,    578,    579,    696,
          708,    713,    744,    767,    773,    779,    834,    848,
          856,    871,    875,    898,    908,    909,    913,    914,
          952,    953,    954,    959,    960,    962,    970,    972,
         1017,   1023,   1062,   1073,   1074,   1153,   1162,   1166,
         1168,   1171,   1174, 302189, 302552, 302555, 302561, 302569,
       302576, 305544])

In [107]:
needs_checking = needs_checking[~needs_checking["gid"].isin(true_matches["gid"].unique())]
mismatched_rows = mismatched_rows[~mismatched_rows["gid"].isin(true_matches["gid"].unique())]

In [108]:
not_matched = df[df["id_2"].isna()]
not_matched

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
49,29.0,53.0,29,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Norrbynrannan uimapaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50,30.0,54.0,30,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Vepon uimapaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,31.0,55.0,31,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Simonbyn uimaranta,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
53,33.0,57.0,33,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Luontokapinetin näkötorni,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,34.0,58.0,34,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Särkijärven päivälaavu ja tulipaikka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1646,NaN,NaN,305821,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Immasten kylän isojako (Immasentie),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1652,NaN,NaN,305825,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Ihminen on muovannut Taimon kylän maisemaa jo ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1657,NaN,NaN,305935,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistysreitin lähtöpiste,startpunkt för friluftsleden,starting point for a recreational route,Tuomiokirkko - pyhiinvaellusreitin aloituspiste,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1658,NaN,NaN,305993,Virkistyskohde,rekreationsobjekt,recreational attraction,Virkistykseen liittyvä erityiskohde,speciellt objekt för friluftsliv,special recreational attraction,Yttilän museokoulu,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [109]:
df = df.merge(not_matched, how='outer', indicator=True)
df = df[df['_merge'] == 'left_only']
df = df.drop(columns=['_merge'])
df

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,6.0,10.0,6,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kuusiston uimaranta,...,Uimapaikka,Kuusiston uimaranta,1.0,0.000000,242140.381600,6.703244e+06,242140.3816,6.703244e+06,NaN,NaN
1,8.0,22.0,8,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Pähkinäisen uimaranta,...,Uimapaikka,Pähkinäisen uimaranta,1.0,1.476903,207317.046300,6.699893e+06,207316.4537,6.699895e+06,NaN,NaN
2,10.0,24.0,10,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Saukkorannan talviuintipaikka,...,Talviuintipaikka,Saukkorannan talviuintipaikka,1.0,0.000000,232689.537400,6.709283e+06,232689.5374,6.709283e+06,NaN,NaN
3,11.0,25.0,11,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Talviuimapaikka,Vinterbadplats,Ice swimming place,Pöylinrannan talviuintipaikka,...,Talviuintipaikka,Pöylinrannan talviuintipaikka,1.0,0.000000,248939.692500,6.699411e+06,248939.6925,6.699411e+06,NaN,NaN
4,12.0,29.0,12,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Järvelän lintulava,...,Luontotorni,Järvelän lintulava,1.0,0.000000,246029.000000,6.711872e+06,246029.0000,6.711872e+06,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,NaN,NaN,305499,Virkistyskohde,rekreationsobjekt,recreational attraction,Tulipaikka,eldplats,campfire site,Grillikatos,...,Kuntosali,Parmaharjun kuntosali,1.0,81.520548,255342.953792,6.718302e+06,255261.5262,6.718298e+06,NaN,NaN
1080,NaN,NaN,305808,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Soinisten kartano (Neljäntammenkuja),...,Luistelukenttä,Luistelukenttä,1.0,177.802291,227141.000000,6.716957e+06,227224.8018,6.717114e+06,NaN,NaN
1083,NaN,NaN,305812,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Markkinatalous muutti maataloutta ja maisemaa ...,...,Luistelukenttä,Luistelukenttä,1.0,191.744847,227412.500000,6.717153e+06,227224.8018,6.717114e+06,NaN,NaN
1086,NaN,NaN,305822,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Markkinatalous muutti maataloutta ja maisemaa ...,...,Luistelukenttä,Luistelukenttä,1.0,191.744847,227412.500000,6.717153e+06,227224.8018,6.717114e+06,NaN,NaN


In [110]:
df['mapped_lipas_category'] = df['class2_fi'].map(mapping_dict)

df['category_match'] = (df['tyyppi_nimi_fi'] == df['mapped_lipas_category']) | (df['tyyppi_nimi_fi'] == df['class2_fi']) | (df['name_fi'] == df['nimi_fi'])

mismatched_rows2 = df[df['category_match'] == False]

if not mismatched_rows2.empty:
    print("Mismatched Categories:")
    display(mismatched_rows2[["name_fi", "nimi_fi", 'class2_fi', 'tyyppi_nimi_fi', 'mapped_lipas_category']])
else:
    print("All categories match!")

Mismatched Categories:


,name_fi,nimi_fi,class2_fi,tyyppi_nimi_fi,mapped_lipas_category
26,Himoisten veneenlaskupaikka,Himoisten sauna ja uimaranta,Veneenlaskupaikka,Uimapaikka,Veneilyn palvelupaikka
27,Kakarlammen uimapaikka,Kakarlammen talviuintipaikka,Uimapaikka,Talviuintipaikka,Uimaranta
28,Tulentekopaikka Hauninen,Haunisten päivätaukokatos,Tulipaikka,"Laavu, kota tai kammi",Ruoanlaitto-/tulentekopaikka
40,Kakarlammen virkistyskeskus,Kakarlammen talviuintipaikka,Tilavuokraus- tai kokouspalvelu,Talviuintipaikka,NaN
41,Novida Cateringservice,Ammattikoulun liikuntasali,Ruokapalvelu,Liikuntasali,NaN
...,...,...,...,...,...
1074,Grillikatos,Parmaharjun kuntosali,Tulipaikka,Kuntosali,Ruoanlaitto-/tulentekopaikka
1080,Soinisten kartano (Neljäntammenkuja),Luistelukenttä,Opastuspiste,Luistelukenttä,Opastuspiste
1083,Markkinatalous muutti maataloutta ja maisemaa ...,Luistelukenttä,Opastuspiste,Luistelukenttä,Opastuspiste
1086,Markkinatalous muutti maataloutta ja maisemaa ...,Luistelukenttä,Opastuspiste,Luistelukenttä,Opastuspiste


In [111]:
needs_checking = pd.concat([needs_checking, mismatched_rows2[mismatched_rows2['mapped_lipas_category'].isna()]], ignore_index=True)
needs_checking

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,45.0,69.0,45,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Kalliorannan laavu,...,Pallokenttä,Karunan jalkapallokenttä,1.0,60.259118,255750.142157,6.690257e+06,255753.2500,6.690197e+06,NaN,False
1,45.0,69.0,45,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Kalliorannan laavu,...,Talviuintipaikka,Kalliorannan talviuintipaikka,2.0,88.229005,255750.142157,6.690257e+06,255706.7500,6.690334e+06,NaN,False
2,62.0,86.0,62,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Virttaan Eräveikkojen maja,...,Kilpahiihtokeskus,Virttaan kilpahiihtokeskus/ Erä-Veikkojen maja,1.0,102.501015,264925.130438,6.766786e+06,265018.9065,6.766745e+06,NaN,False
3,62.0,86.0,62,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Virttaan Eräveikkojen maja,...,Ampumarata,Virttaan ampumarata,2.0,112.938936,264925.130438,6.766786e+06,264874.1761,6.766685e+06,NaN,False
4,158.0,183.0,158,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Sauna,bastu,sauna,Lehmijärven sauna,...,Lähiliikuntapaikka,Lehmijärven uimarannan leikkipaikka,1.0,29.351507,283182.500010,6.691274e+06,283159.2000,6.691291e+06,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,NaN,NaN,302554,Virkistyskohde,rekreationsobjekt,recreational attraction,Melontareitin lähtöpiste,startpunkt för paddlingsleden,starting point for a paddling route,Stora Masugnsträsketin kanoottilaituri ja vuok...,...,Veneilyn palvelupaikka,Stora Masugnsträsketin kanoottilaituri,1.0,3.798501,249627.250000,6.663014e+06,249628.8783,6.663011e+06,NaN,False
213,NaN,NaN,302560,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,Ekniemen luontopolkujen pysäköintopaikka,...,Opastuspiste,Ekniemen luontopolkujen opastaulu,1.0,29.227219,256389.000000,6.683575e+06,256385.2504,6.683546e+06,NaN,False
214,NaN,NaN,302573,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,Invapysäköinti,...,Koiraurheilualue,Kemiön koirapuisto,1.0,122.483344,262365.333313,6.677981e+06,262248.9161,6.677943e+06,NaN,False
215,NaN,NaN,305485,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,"Parmaharjun luontopolku, parkkipaikka",...,Kuntosali,Parmaharjun kuntosali,1.0,50.427351,255311.894240,6.718301e+06,255261.5262,6.718298e+06,NaN,False


In [112]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2['mapped_lipas_category'].isna()]
mismatched_rows2

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
26,41.0,65.0,41,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Veneenlaskupaikka,Sjösättningsramp,Slipway,Himoisten veneenlaskupaikka,...,Uimapaikka,Himoisten sauna ja uimaranta,1.0,46.082469,204049.808172,6.739157e+06,204025.7500,6.739118e+06,Veneilyn palvelupaikka,False
27,43.0,67.0,43,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kakarlammen uimapaikka,...,Talviuintipaikka,Kakarlammen talviuintipaikka,1.0,100.264676,277568.324552,6.740343e+06,277488.1228,6.740403e+06,Uimaranta,False
28,44.0,68.0,44,Virkistyskohde,rekreationsobjekt,recreational attraction,Tulipaikka,eldplats,campfire site,Tulentekopaikka Hauninen,...,"Laavu, kota tai kammi",Haunisten päivätaukokatos,1.0,89.339591,236429.000000,6.717008e+06,236498.0000,6.716951e+06,Ruoanlaitto-/tulentekopaikka,False
44,65.0,89.0,65,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Vieraslaituri,Gästbrygga,Visitors' berth,Vestlaxin vieraslaituri,...,Uimapaikka,Vestlaxin uimapaikka,1.0,112.217777,264848.642157,6.667092e+06,264737.9187,6.667110e+06,Veneilyn palvelupaikka,False
55,77.0,101.0,77,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ölmosin uimaranta,...,Uimapaikka,Västerbuktenin uimaranta,1.0,144.402567,243804.000000,6.670654e+06,243659.6505,6.670650e+06,Uimaranta,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1074,NaN,NaN,305499,Virkistyskohde,rekreationsobjekt,recreational attraction,Tulipaikka,eldplats,campfire site,Grillikatos,...,Kuntosali,Parmaharjun kuntosali,1.0,81.520548,255342.953792,6.718302e+06,255261.5262,6.718298e+06,Ruoanlaitto-/tulentekopaikka,False
1080,NaN,NaN,305808,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Soinisten kartano (Neljäntammenkuja),...,Luistelukenttä,Luistelukenttä,1.0,177.802291,227141.000000,6.716957e+06,227224.8018,6.717114e+06,Opastuspiste,False
1083,NaN,NaN,305812,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Markkinatalous muutti maataloutta ja maisemaa ...,...,Luistelukenttä,Luistelukenttä,1.0,191.744847,227412.500000,6.717153e+06,227224.8018,6.717114e+06,Opastuspiste,False
1086,NaN,NaN,305822,Virkistyskohde,rekreationsobjekt,recreational attraction,Opastuspiste,informationspunkt,information point,Markkinatalous muutti maataloutta ja maisemaa ...,...,Luistelukenttä,Luistelukenttä,1.0,191.744847,227412.500000,6.717153e+06,227224.8018,6.717114e+06,Opastuspiste,False


In [113]:
true_matches = pd.concat([true_matches, df[df['category_match'] == True]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,NaN,NaN,302557,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Merikruunun vierassatama,...,Veneilyn palvelupaikka,Merikruunun vierassatama,1.0,26.810498,248734.000000,6.668594e+06,248760.7938,6.668595e+06,Luontotorni,True
264,NaN,NaN,302562,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Biskopsön luontotorni,...,Luontotorni,Biskopsön näköalatorni,1.0,16.376500,251044.000000,6.657750e+06,251030.2591,6.657759e+06,Luontotorni,True
265,NaN,NaN,302563,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Vänön uimaranta,...,Uimapaikka,Vänön uimapaikka,1.0,9.013878,232011.000000,6.646822e+06,232016.0000,6.646830e+06,Uimaranta,True
266,NaN,NaN,302565,Virkistyskohde,rekreationsobjekt,recreational attraction,Yöpymislaavu tai -kota,övernattningsvindskydd eller -kåta,lean-to or lap pole tent for overnighting,Sundsvedjan laavu,...,"Laavu, kota tai kammi",Sundsvedjan laavu,1.0,11.926860,244632.000000,6.663120e+06,244638.5000,6.663130e+06,NaN,True


In [114]:
mismatched_rows2[mismatched_rows2['class2_fi'] == "Uimaranta"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
55,77.0,101.0,77,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ölmosin uimaranta,...,Uimapaikka,Västerbuktenin uimaranta,1.0,144.402567,243804.000000,6.670654e+06,243659.6505,6.670650e+06,Uimaranta,False
861,1022.0,1049.0,1022,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Bläsnäs,...,Uimapaikka,Bläsnäsin uimala,1.0,3.393387,240263.513141,6.695582e+06,240261.5029,6.695585e+06,Uimaranta,False


In [115]:
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Ölmosin uimaranta") & (mismatched_rows2["nimi_fi"] == "Västerbuktenin uimaranta")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Bläsnäs") & (mismatched_rows2["nimi_fi"] == "Bläsnäsin uimala")]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,NaN,NaN,302563,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Vänön uimaranta,...,Uimapaikka,Vänön uimapaikka,1.0,9.013878,232011.000000,6.646822e+06,232016.0000,6.646830e+06,Uimaranta,True
266,NaN,NaN,302565,Virkistyskohde,rekreationsobjekt,recreational attraction,Yöpymislaavu tai -kota,övernattningsvindskydd eller -kåta,lean-to or lap pole tent for overnighting,Sundsvedjan laavu,...,"Laavu, kota tai kammi",Sundsvedjan laavu,1.0,11.926860,244632.000000,6.663120e+06,244638.5000,6.663130e+06,NaN,True
267,NaN,NaN,305459,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Pitkäjärven uimaranta,...,Uimapaikka,Pitkäjärven uimaranta,1.0,43.219918,297453.560640,6.731412e+06,297473.0000,6.731450e+06,Uimaranta,True
268,77.0,101.0,77,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ölmosin uimaranta,...,Uimapaikka,Västerbuktenin uimaranta,1.0,144.402567,243804.000000,6.670654e+06,243659.6505,6.670650e+06,Uimaranta,False


In [116]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2["gid"].isin(true_matches["gid"].unique())]

In [117]:

true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["name_fi"] == "Maarian Mahdin ulkoilumaja") & (mismatched_rows2["nimi_fi"] == "Maarian mahti ry ulkoilumaja")]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266,NaN,NaN,302565,Virkistyskohde,rekreationsobjekt,recreational attraction,Yöpymislaavu tai -kota,övernattningsvindskydd eller -kåta,lean-to or lap pole tent for overnighting,Sundsvedjan laavu,...,"Laavu, kota tai kammi",Sundsvedjan laavu,1.0,11.926860,244632.000000,6.663120e+06,244638.5000,6.663130e+06,NaN,True
267,NaN,NaN,305459,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Pitkäjärven uimaranta,...,Uimapaikka,Pitkäjärven uimaranta,1.0,43.219918,297453.560640,6.731412e+06,297473.0000,6.731450e+06,Uimaranta,True
268,77.0,101.0,77,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ölmosin uimaranta,...,Uimapaikka,Västerbuktenin uimaranta,1.0,144.402567,243804.000000,6.670654e+06,243659.6505,6.670650e+06,Uimaranta,False
269,1022.0,1049.0,1022,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Bläsnäs,...,Uimapaikka,Bläsnäsin uimala,1.0,3.393387,240263.513141,6.695582e+06,240261.5029,6.695585e+06,Uimaranta,False


In [118]:
true_matches = pd.concat([true_matches, mismatched_rows2[(mismatched_rows2["gid"] == 302162) & (mismatched_rows2["nimi_fi"] == "Palmusen puiston frisbeegolfrata")]], ignore_index=True)
true_matches

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,1062.0,1089.0,1062,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Poikon uimapaikka,...,Uimapaikka,Poikon uimapaikka,3.0,10.932850,217581.250000,6.709144e+06,217570.5514,6.709147e+06,Uimaranta,True
1,1150.0,1177.0,1150,Virkistyskohde,rekreationsobjekt,recreational attraction,Luonto- tai lintutorni,natur- eller fågeltorn,nature or bird watching tower,Järvelän näköala- ja lintutorni,...,Luontotorni,Järvelän näköala- ja lintutorni,1.0,5.098060,246289.500000,6.711695e+06,246290.3762,6.711690e+06,Luontotorni,True
2,133.0,158.0,133,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ekvallan uimaranta,...,Uimaranta,Ekvallan uimaranta,1.0,0.000057,236944.259856,6.703199e+06,236944.2598,6.703199e+06,Uimaranta,True
3,36.0,60.0,36,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Kalikan uimaranta,...,Uimapaikka,Pöytyän Kalikan uimaranta,2.0,43.942079,250980.466581,6.761814e+06,250977.2504,6.761770e+06,Uimaranta,True
4,53.0,77.0,53,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Tilavuokraus- tai kokouspalvelu,Utrymmes- eller konferensservice,Facility rental or conference service,Suojalan hiihtomaja,...,Ulkoilumaja/hiihtomaja,Suojalan hiihtomaja,2.0,71.686274,264555.572829,6.694371e+06,264484.2094,6.694365e+06,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,NaN,NaN,305459,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Pitkäjärven uimaranta,...,Uimapaikka,Pitkäjärven uimaranta,1.0,43.219918,297453.560640,6.731412e+06,297473.0000,6.731450e+06,Uimaranta,True
268,77.0,101.0,77,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Ölmosin uimaranta,...,Uimapaikka,Västerbuktenin uimaranta,1.0,144.402567,243804.000000,6.670654e+06,243659.6505,6.670650e+06,Uimaranta,False
269,1022.0,1049.0,1022,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimaranta,Badstrand,Beach,Bläsnäs,...,Uimapaikka,Bläsnäsin uimala,1.0,3.393387,240263.513141,6.695582e+06,240261.5029,6.695585e+06,Uimaranta,False
270,80.0,104.0,80,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Maarian Mahdin ulkoilumaja,...,Kilpahiihtokeskus,Maarian mahti ry ulkoilumaja,1.0,0.000083,244801.900070,6.717860e+06,244801.9000,6.717860e+06,Ulkoilumaja/hiihtomaja,False


In [119]:
mismatched_rows2 = mismatched_rows2[~mismatched_rows2["gid"].isin(true_matches["gid"].unique())]

In [120]:
mismatched_rows2[mismatched_rows2["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

array([], dtype=int64)

In [121]:
original_df = pd.read_csv("../../Data/really_3_NN_virma_lipas.csv")

In [122]:
from rapidfuzz import fuzz

for i, (name1, name2) in enumerate(zip(mismatched_rows2["name_fi"].tolist(), mismatched_rows2["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 85:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

Different at index 0: 'Himoisten veneenlaskupaikka' ≠ 'Himoisten sauna ja uimaranta' (score: 58.18181818181818)
Different at index 1: 'Kakarlammen uimapaikka' ≠ 'Kakarlammen talviuintipaikka' (score: 80.0)
Different at index 2: 'Tulentekopaikka Hauninen' ≠ 'Haunisten päivätaukokatos' (score: 53.06122448979591)
Different at index 3: 'Vestlaxin vieraslaituri' ≠ 'Vestlaxin uimapaikka' (score: 65.11627906976744)
Different at index 4: 'Tupurin reittien info- ja lähtöpiste' ≠ 'Tupurin maja' (score: 41.666666666666664)
Different at index 5: 'Lakjärven opastaulu' ≠ 'Lakjärven laavu' (score: 76.47058823529412)
Different at index 6: 'Rantapiha' ≠ 'Savojärven uimapaikka' (score: 40.0)
Different at index 7: 'Kuustonlahden lintutornin pysäköintialue' ≠ 'Kuustonlahden luontolava, Mietoistenlahti' (score: 59.25925925925925)
Different at index 8: 'Teijo Ski & Action Park' ≠ 'Meri-Teijon pyöräilyalue' (score: 29.78723404255319)
Different at index 9: 'Vajosuon vuokratupa' ≠ 'Vajosuon laavu' (score: 66.6

In [123]:
original_df["tyyppi_nimi_fi"].value_counts()

tyyppi_nimi_fi
Uimapaikka                            162
Laavu, kota tai kammi                  85
Opastuspiste                           72
Talviuintipaikka                       65
Veneilyn palvelupaikka                 54
Luontotorni                            53
Uimaranta                              53
Ulkokuntoilupaikka                     45
Frisbeegolfrata                        38
Beachvolley-/rantalentopallokenttä     38
Pallokenttä                            20
Ruoanlaitto-/tulentekopaikka           20
Liikuntasali                           20
Kalastuskohde (piste)                  19
Telttailu ja leiriytyminen             14
Tenniskenttä                           13
Kuntosali                              12
Luistelukenttä                         11
Koripallokenttä                         8
Lähiliikuntapaikka                      7
Kuntokeskus                             6
Kilpahiihtokeskus                       6
Yleisurheilun harjoitusalue             5
Lentopallokenttä   

In [124]:
original_df["class2_fi"].value_counts()

class2_fi
Uimapaikka                               165
Opastuspiste                             159
Retkeilyä palveleva parkkipaikka         130
Luonnonmuistomerkki tai näköalapaikka    127
Yhteysaluslaituri                        122
Retki- tai luonnonsatama                  99
Luonto- tai lintutorni                    87
Vierassatama                              76
Tulipaikka                                69
Yleisö-wc tai -puucee                     53
Talviuimapaikka                           53
Yöpymislaavu tai -kota                    45
Vieraslaituri                             44
Päivälaavu tai -kota                      43
Uimaranta                                 37
Leirintä- tai caravanalue                 31
Leirikeskus                               30
Kulttuuripalvelu                          28
Virkistysreitin lähtöpiste                25
Virkistykseen liittyvä erityiskohde       22
Majoituspalvelu                           19
Telttailupaikka                           17


SEURAAVAKSI: <br>
mismatched_rows2 chekattu ja oikeat matsit lisätty true_matches. Kaikki siellä olevat kohteet on mätsätty väärin eli täytyy laittaa niille, ettei mätsiä löytynyt ja lisätä not_matched. <br>
vielä jäljellä needs_checking ja mismatched rows

In [125]:
needs_checking.shape

(217, 62)

In [126]:
mismatched_rows.shape

(138, 62)

In [127]:
import numpy as np

#mismatched_rows2[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
#mismatched_rows2

In [128]:
not_matched = pd.concat([not_matched, mismatched_rows2], axis=0, ignore_index=True)

In [129]:
for i, (name1, name2) in enumerate(zip(mismatched_rows["name_fi"].tolist(), mismatched_rows["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 85:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

Different at index 0: 'Hiekkahelmen talviuintipaikka' ≠ 'Hiekkahelmi' (score: 55.00000000000001)
Different at index 1: 'Hiekkahelmen talviuintipaikka' ≠ 'Hiekkahelmen beachvolleykenttä' (score: 57.6271186440678)
Different at index 2: 'Hakkenpään laavupolun infotaulu' ≠ 'Hakkenpään uimapaikka' (score: 57.692307692307686)
Different at index 3: 'Hakkenpään laavupolun infotaulu' ≠ 'Hakkenpää' (score: 44.99999999999999)
Different at index 4: 'Hovimäki Camping' ≠ 'Hovimäen uimapaikka' (score: 40.0)
Different at index 5: 'Hovimäki Camping' ≠ 'Hovimäen leirintäalueen lentopallokenttä' (score: 25.0)
Different at index 6: 'Kangastupa' ≠ 'Oripään ampumahiihtokeskus' (score: 22.22222222222222)
Different at index 7: 'Kangastupa' ≠ 'Oripään frisbeegolfrata' (score: 30.303030303030297)
Different at index 8: 'Kangastupa' ≠ 'Kangastuvan kuntoportaat' (score: 58.82352941176471)
Different at index 9: 'Matikan kuntoradan ja frisbeegolfradan opastuspiste' ≠ 'Laitilan frisbeegolfrata' (score: 56.00000000000

'Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin hiihtokeskus' <br>
'Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin urheilutalo <br>
Auran kunnan Koivuniemen virkistysalue' ≠ 'Koivuniemen virkistysalue <br>
Högsåran Sandvikuddenin tulipaikka	...	Ruoanlaitto-/tulentekopaikka	Högsåran tulentekopaikka
Höglandin tulipaikka	...	Ruoanlaitto-/tulentekopaikka	Höglandin tulentekopaikka
Match at index 124: 'Taalintehdas DiscGolf Park' ~ 'Taalintehdas DiscGolfPark' (score: 98.0392156862745)




In [130]:
for i, (name1, name2) in enumerate(zip(needs_checking["name_fi"].tolist(), needs_checking["nimi_fi"].tolist())):
    score = fuzz.token_sort_ratio(name1, name2)
    if score > 75:
        print(f"Match at index {i}: '{name1}' ~ '{name2}' (score: {score})")
    else:
        print(f"Different at index {i}: '{name1}' ≠ '{name2}' (score: {score})")

Different at index 0: 'Kalliorannan laavu' ≠ 'Karunan jalkapallokenttä' (score: 47.61904761904761)
Different at index 1: 'Kalliorannan laavu' ≠ 'Kalliorannan talviuintipaikka' (score: 68.08510638297872)
Different at index 2: 'Virttaan Eräveikkojen maja' ≠ 'Virttaan kilpahiihtokeskus/ Erä-Veikkojen maja' (score: 69.44444444444444)
Different at index 3: 'Virttaan Eräveikkojen maja' ≠ 'Virttaan ampumarata' (score: 53.333333333333336)
Different at index 4: 'Lehmijärven sauna' ≠ 'Lehmijärven uimarannan leikkipaikka' (score: 61.53846153846154)
Match at index 5: 'Lehmijärven sauna' ~ 'Lehmijärven uimaranta' (score: 78.94736842105263)
Different at index 6: 'Lehmijärven sauna' ≠ 'Lehmijärven beachvolleykenttä 1' (score: 58.33333333333333)
Different at index 7: 'Kuru-reitistön parkkipaikka' ≠ 'Beachvolleykenttä' (score: 13.636363636363635)
Different at index 8: 'Kuru-reitistön parkkipaikka' ≠ 'Porhonkallion uimapaikka' (score: 43.13725490196079)
Different at index 9: 'Urheilupuiston pysäköintial

tru matches pitää poistaa hakkenpää

mis_matched_rows ja mis_matched_rows2 chekattu needs chenking tarvii vielä ja kerää kaikki kiikun kaakun olevat pisteet johonkin. 

In [131]:
mismatched_rows[mismatched_rows["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
184,505.0,532.0,505,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Valpperin hiihtokeskus ja urheilutalo,...,Kilpahiihtokeskus,Valpperin hiihtokeskus,1.0,0.000068,241542.685935,6.734939e+06,241542.6859,6.734939e+06,Ulkoilumaja/hiihtomaja,False
185,505.0,532.0,505,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Valpperin hiihtokeskus ja urheilutalo,...,Liikuntasali,Valpperin urheilutalo,2.0,21.513360,241542.685935,6.734939e+06,241522.0000,6.734933e+06,Ulkoilumaja/hiihtomaja,False
186,505.0,532.0,505,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Valpperin hiihtokeskus ja urheilutalo,...,Frisbeegolfrata,Valpperin frisbeegolfrata,3.0,29.105412,241542.685935,6.734939e+06,241545.3508,6.734968e+06,Ulkoilumaja/hiihtomaja,False


In [132]:
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Auran kunnan Koivuniemen virkistysalue") & (mismatched_rows["nimi_fi"] == "Koivuniemen virkistysalue")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Taalintehdas DiscGolf Park") & (mismatched_rows["nimi_fi"] == "Taalintehdas DiscGolfPark")]], ignore_index=True)
true_matches = pd.concat([true_matches, mismatched_rows[(mismatched_rows["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo") & (mismatched_rows["nimi_fi"] == "Valpperin hiihtokeskus")]], ignore_index=True)

In [133]:
true_matches[true_matches["name_fi"]== "Valpperin hiihtokeskus ja urheilutalo"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
274,505.0,532.0,505,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Valpperin hiihtokeskus ja urheilutalo,...,Kilpahiihtokeskus,Valpperin hiihtokeskus,1.0,0.000068,241542.685935,6.734939e+06,241542.6859,6.734939e+06,Ulkoilumaja/hiihtomaja,False


In [134]:
mismatched_rows[mismatched_rows["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

array([   505,   1041, 302564])

In [135]:
mismatched_rows = mismatched_rows[~mismatched_rows["gid"].isin(true_matches["gid"].unique())]

In [136]:
mismatched_rows = mismatched_rows.drop_duplicates(subset='gid', keep='first')

In [137]:
#mismatched_rows[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan

In [138]:
not_matched = pd.concat([not_matched, mismatched_rows], axis=0, ignore_index=True)

Himoisten rantasauna' ≠ 'Himoisten sauna ja uimaranta meillä kaksi eri pistettä, lippassa yksi<br>
- toteutus:
    - virman pisteet himoisten rantasauna ja himoisten uimaranta molemmat mätsättiin lippaan Himoisten sauna ja uimaranta pisteeseen

Rautilan rantasauna' ≠ 'Rautilan sauna ja uimapaikka meillä kaksi pistettä lippaassa yksi <br>
- toteutus:
    - virman pisteet rautilan rantasauna ja rautilan uimaranta molemmat mätsättiin lippaan Rautilan sauna ja uimapaikka pisteeseen

Särkijärven rantasauna	...	Uimapaikka	Särkijärven uimapaikka  -virmassa myös särkijärven uimaranta ja rantasauna <br>
- toteutus:
    - särkijärven rantasauna ei matchiä, koska lippaassa ei ole sitä, särkijärven uimaranta ja rantasauna mätsätty särkijärven uimapaikkaan

Aseman monitoimikenttä, jääkaukalo' ≠ 'Aseman kentän monitoimikaukalo' <br>
Aseman monitoimikenttä, jääkaukalo' ≠ 'Aseman koulun luistelukenttä <br>
- toteutus:
    - Aseman monitoimikenttä, jääkaukalo' = 'Aseman kentän monitoimikaukalo, koska more similar
    - Valpperin hiihtokeskus ja urheilutalo' ≠ 'Valpperin hiihtokeskus',  koska more similar <br>

In [139]:
true_matches[true_matches["name_fi"] == "Valpperin hiihtokeskus ja urheilutalo"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,...,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
274,505.0,532.0,505,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Ulkoilu- tai hiihtomaja,Frilufts- eller skidstuga,Outdoor or ski cabin,Valpperin hiihtokeskus ja urheilutalo,...,Kilpahiihtokeskus,Valpperin hiihtokeskus,1.0,0.000068,241542.685935,6.734939e+06,241542.6859,6.734939e+06,Ulkoilumaja/hiihtomaja,False


In [140]:
true_matches.shape

(275, 62)

In [141]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Vaarniemen laavu ja näkötorni") & (needs_checking["nimi_fi"] == "Vaarniemen näkötorni")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Amospuiston kuntoportaat") & (needs_checking["nimi_fi"] == "Kuntoportaat")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Västanfjärds DiscGolfPark") & (needs_checking["nimi_fi"] == "Västanfjärd DiscGolfPark")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Kuntoportaat") & (needs_checking["nimi_fi"] == "Saukonojan kuntoportaat")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Virttaan Eräveikkojen maja") & (needs_checking["nimi_fi"] == "Virttaan kilpahiihtokeskus/ Erä-Veikkojen maja")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Kavilannummen mottorirata") & (needs_checking["nimi_fi"] == "Kavilannummen motocross-rata")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Agilityrata") & (needs_checking["nimi_fi"] == "MynSK:n koulutuskenttä")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Stora Masugnsträsketin kanoottilaituri ja vuokrakanootit") & (needs_checking["nimi_fi"] == "Stora Masugnsträsketin kanoottilaituri")]], ignore_index=True)

In [142]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Rautilan rantasauna") & (needs_checking["nimi_fi"] == "Rautilan sauna ja uimapaikka")]], ignore_index=True)

In [143]:
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Himoisten rantasauna") & (needs_checking["nimi_fi"] == "Himoisten sauna ja uimaranta")]], ignore_index=True)
true_matches = pd.concat([true_matches, needs_checking[(needs_checking["name_fi"] == "Aseman monitoimikenttä, jääkaukalo") & (needs_checking["nimi_fi"] == "Aseman kentän monitoimikaukalo")]], ignore_index=True)

In [144]:
true_matches.shape

(285, 62)

In [145]:
needs_checking[needs_checking["gid"].isin(true_matches["gid"].unique())]["gid"].unique()

array([    62,    340, 302291, 302293, 302377, 302390, 302570,    401,
          489, 302554])

In [146]:
needs_checking.shape

(217, 62)

In [147]:
needs_checking = needs_checking[~needs_checking["gid"].isin(true_matches["gid"].unique())]

In [148]:
needs_checking = needs_checking.drop_duplicates(subset='gid', keep='first')

In [149]:
#needs_checking[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan

In [150]:
not_matched = pd.concat([not_matched, needs_checking], axis=0, ignore_index=True)

In [151]:
pd.set_option("display.max_columns", None)

not_matched[not_matched.duplicated(subset='name_fi', keep=False)]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
123,264.0,289.0,264,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Yhteysaluslaituri,Förbindelsebåtsbrygga,Ferry harbour,Nåtö,NaN,NaN,osoitteeton,Parainen,Turunmaa,Varsinais-Suomi,Yhteysaluslaituria saa käyttää tilapäiseen kii...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yksityinen,"Briitta Åkerberg, Torvald Åkerberg kuolinpesä",ELY-keskus,NaN,satunnainen,NaN,1899/12/30,NaN,LN,177765.952187,6.703704e+06,2017/11/08,21760.0,573,21,2,NaN,T,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
443,706.0,733.0,706,Virkistyskohde,rekreationsobjekt,recreational attraction,Päivälaavu tai -kota,dagsvindskydd eller -kåta,lean-to or lap pole tent for daytime use,Tuorlan Majatalo,NaN,NaN,Tuorlantie 1 E,Kaarina,Turku,Varsinais-Suomi,"Rantasauna, jossa kaksi laavua, nuotiopaikka /...",NaN,NaN,NaN,Esteetön,"polttopuut, katettu laavu",www.tuorlanmajatalo.fi,NaN,NaN,+358401234567,erkki@esimerkki.fi,Kunta,Marja Vaiste,Ammattiopisto Livia,NaN,jatkuva,NaN,2020/02/28,NaN,"TuorlanMajatalo, kukkahammar",249119.000000,6.705760e+06,2020/02/28,21500.0,202,23,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486,781.0,808.0,781,Matkailupalvelukohde,Objekt för turismfunktioner,Tourist attraction,Leirikeskus,Lägercenter,Summer camp centre,Kesä-Angelan kesäkoti,NaN,NaN,Angelantie 77,Salo,Salon seutu,Varsinais-Suomi,Salon seurakunnan kesäkoti Angelniemessä.,NaN,NaN,NaN,NaN,NaN,https://www.salonseurakunta.fi/00010264-kesa-a...,NaN,NaN,NaN,NaN,Seurakunta,Salon seurakunta,Salon seurakunta,NaN,jatkuva,NaN,1899/12/30,NaN,"LN, TerhiSalosta, SaloMK",277108.439612,6.692913e+06,2018/01/11,25230.0,734,22,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
506,808.0,835.0,808,Virkistyskohde,rekreationsobjekt,recreational attraction,Retkeilyä palveleva parkkipaikka,parkeringsplats för vandring,parking lot for hikers,Kyhkärännokan pysäköintialue,NaN,NaN,NaN,Pyhäranta,Vakka-Suomi,Varsinais-Suomi,Hierkonpolun luoteispään p-alue. Hierkonpolku ...,NaN,NaN,NaN,NaN,NaN,https://reidu.yhdistysavain.fi/hierkonpolku/,NaN,NaN,NaN,NaN,Valtio,Suomen valtio,NaN,NaN,ei ylläpitoa,NaN,2018/10/18,NaN,"LN, TA",193385.250000,6.780316e+06,2018/12/10,27340.0,631,24,2,Leader-hanke,T,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
523,825.0,852.0,825,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Retkeilyä palveleva parkkipaikka,Parkeringsplats för vandring,Parking lot for hikers,Särkijärven parkkipaikka,NaN,NaN,Särkijärventie 2,Pöytyä,Loimaan seutu,Varsinais-Suomi,Kuhankuonon retkeilyreitistöön liittyvä kohde.,NaN,NaN,NaN,NaN,p-paikka,http://www.kuhankuono.fi,NaN,NaN,NaN,NaN,Yhdistys,Leijankorven yhteismetsä ry,Kuhankuonon retkeilyreitistö- ja virkistysalue...,NaN,jatkuva,NaN,1899/12/30,NaN,LN,246115.694698,6.747004e+06,2018/01/12,21900.0,636,25,2,NaN,F,T,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1142.0,1169.0,1142,Matkailupalvelukohde,objekt för turismfunktioner,tourist attraction,Tilavuokraus- tai kokouspalvelu,utrymmes- eller konferensservice,facility rental and conference service,Tuorlan Majatalo,NaN,NaN,Tuorl

In [152]:
df1_gids = set(true_matches["gid"])
df2_gids = set(not_matched["gid"])
df3_gids = set(original_df["gid"])

missing_from_1_and_2 = df3_gids - (df1_gids | df2_gids)

original_df[original_df["gid"].isin(missing_from_1_and_2)]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y


In [153]:
##not_matched = not_matched.drop_duplicates(subset=["name_fi", "x_eureffin", "y_eureffin"], keep="first")

In [154]:
not_matched.shape

(1034, 62)

In [155]:
true_matches.shape

(285, 62)

In [156]:


not_matched[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
not_matched.head()

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,29.0,53.0,29,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Norrbynrannan uimapaikka,NaN,NaN,Tuulenpuoli 5,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,241444.642157,6.696252e+06,2017/04/05,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30.0,54.0,30,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Vepon uimapaikka,NaN,NaN,Veponranta 17,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yritys,NordKalk Oyj,ei ylläpitoa,ei ylläpitoa,ei ylläpitoa,NaN,NaN,NaN,LN,239880.642157,6.696266e+06,2017/04/04,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31.0,55.0,31,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Simonbyn uimaranta,NaN,NaN,Harjulantie 120,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,225128.077680,6.684804e+06,2017/04/06,21650.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,33.0,57.0,33,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Luontokapinetin näkötorni,NaN,NaN,Hovilanmäentie 2,Pöytyä,Loimaan seutu,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Pöytyän kunta,ei tietoa ylläpitäjästä,NaN,ei tietoa ylläpitoluokasta,NaN,2016/11/01,NaN,"LN, Luontokapinetti",251422.638706,6.757558e+06,2017/04/07,21900.0,636,25,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,34.0,58.0,34,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Särkijärven päivälaavu ja tulipaikka,NaN,NaN,Särkijärventie 21,Pöytyä,Loimaan seutu,Varsinais-Suomi,Kuhankuonon retkeilyreitistöön kuuluva kohde.,NaN,NaN,NaN,NaN,"puucee, laavu ja nuotiopaikka",www.kuhankuono.fi,NaN,NaN,NaN,NaN,Yhdistys,Leijankorven yhteismetsä ry,Kuhankuonon retkeilyreitistö- ja virkistysalue...,http://www.kuhankuono.fi/fi/palaute/,jatkuva,NaN,NaN,NaN,LN,246257.773768,6.747123e+06,2017/04/04,21900.0,636,25,2,NaN,F,T,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [157]:
df1_gids = set(true_matches["gid"])
df2_gids = set(not_matched["gid"])
df3_gids = set(original_df["gid"])

missing_from_1_and_2 = df3_gids - (df1_gids | df2_gids)

original_df[original_df["gid"].isin(missing_from_1_and_2)].shape

(0, 60)

Väärät mätsit true_matchessä <br>
Poistettu ja lisätty not matched <br>

Vuohensaaren luontopolun lähtöpiste		Vuohensaaren luontopolun opastuspiste <br>
Hakkenpää	Hakkenpää  <br>
Yxskärin kiinnittymispaikka 3	Retki- tai luonnonsatama	Yxskärin retkisatama	<br>
Yxskärin kiinnittymispaikka 2	Retki- tai luonnonsatama	Yxskärin retkisatama	<br>
Höglandin kiinnityslenkki	Retki- tai luonnonsatama	Höglandin retkisatama <br>
Höglandin kiinnityslenkki 2	Retki- tai luonnonsatama	Höglandin retkisatama <br>
Ekniemen lomakylä	Majoituspalvelu	Ekniemen lomakylä <br>
Lakjärven laavu 1	Yöpymislaavu tai -kota	Lakjärven laavu <br>
Matildajärventien pään opastaulu	Opastuspiste	Teijon luontotalo	Opastuspiste	Opastuspiste <br>
238	Matildajärven pysäköintialueen opastaulu	Opastuspiste	Teijon luontotalo	<br>


In [158]:
true_matches[["name_fi", "class2_fi", "nimi_fi", "tyyppi_nimi_fi", "mapped_lipas_category"]]

,name_fi,class2_fi,nimi_fi,tyyppi_nimi_fi,mapped_lipas_category
0,Poikon uimapaikka,Uimapaikka,Poikon uimapaikka,Uimapaikka,Uimaranta
1,Järvelän näköala- ja lintutorni,Luonto- tai lintutorni,Järvelän näköala- ja lintutorni,Luontotorni,Luontotorni
2,Ekvallan uimaranta,Uimaranta,Ekvallan uimaranta,Uimaranta,Uimaranta
3,Kalikan uimaranta,Uimapaikka,Pöytyän Kalikan uimaranta,Uimapaikka,Uimaranta
4,Suojalan hiihtomaja,Tilavuokraus- tai kokouspalvelu,Suojalan hiihtomaja,Ulkoilumaja/hiihtomaja,NaN
...,...,...,...,...,...
280,Agilityrata,Virkistykseen liittyvä erityiskohde,MynSK:n koulutuskenttä,Koiraurheilualue,NaN
281,Stora Masugnsträsketin kanoottilaituri ja vuok...,Melontareitin lähtöpiste,Stora Masugnsträsketin kanoottilaituri,Veneilyn palvelupaikka,NaN
282,Rautilan rantasauna,Sauna,Rautilan sauna ja uimapaikka,Uimapaikka,NaN
283,Himoisten rantasauna,Sauna,Himoisten sauna ja uimaranta,Uimapaikka,NaN


In [159]:
true_matches.shape

(285, 62)

In [160]:
not_matched.shape

(1034, 62)

In [161]:
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Vuohensaaren luontopolun lähtöpiste") & (true_matches["nimi_fi"] == "Vuohensaaren luontopolun opastuspiste")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Hakkenpää") & (true_matches["nimi_fi"] == "Hakkenpää")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Yxskärin kiinnittymispaikka 3") & (true_matches["nimi_fi"] == "Yxskärin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Yxskärin kiinnittymispaikka 2") & (true_matches["nimi_fi"] == "Yxskärin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Höglandin kiinnityslenkki") & (true_matches["nimi_fi"] == "Höglandin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Höglandin kiinnityslenkki 2") & (true_matches["nimi_fi"] == "Höglandin retkisatama")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Lakjärven laavu 1") & (true_matches["nimi_fi"] == "Lakjärven laavu")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Matildajärventien pään opastaulu") & (true_matches["nimi_fi"] == "Teijon luontotalo")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Matildajärven pysäköintialueen opastaulu") & (true_matches["nimi_fi"] == "Teijon luontotalo")]], ignore_index=True)
not_matched = pd.concat([not_matched, true_matches[(true_matches["name_fi"] == "Ekniemen lomakylä") & (true_matches["nimi_fi"] == "Ekniemen lomakylä") & (true_matches["class2_fi"] == "Majoituspalvelu")]], ignore_index=True)

In [162]:
not_matched.shape

(1043, 62)

In [163]:
true_matches[true_matches["gid"].isin(not_matched["gid"].unique())]["gid"].unique()

array([106, 348, 457, 456, 465, 466, 265, 948, 949])

In [164]:
true_matches = true_matches[~true_matches["gid"].isin(not_matched["gid"].unique())]

In [165]:
true_matches.shape

(276, 62)

In [166]:
not_matched[['id_2', 'tyyppi_nimi_fi', 'nimi_fi', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'mapped_lipas_category', 'category_match']] = np.nan
not_matched

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,29.0,53.0,29,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Norrbynrannan uimapaikka,NaN,NaN,Tuulenpuoli 5,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,241444.642157,6.696252e+06,2017/04/05,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30.0,54.0,30,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Vepon uimapaikka,NaN,NaN,Veponranta 17,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yritys,NordKalk Oyj,ei ylläpitoa,ei ylläpitoa,ei ylläpitoa,NaN,NaN,NaN,LN,239880.642157,6.696266e+06,2017/04/04,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31.0,55.0,31,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Simonbyn uimaranta,NaN,NaN,Harjulantie 120,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,225128.077680,6.684804e+06,2017/04/06,21650.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,33.0,57.0,33,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Luontokapinetin näkötorni,NaN,NaN,Hovilanmäentie 2,Pöytyä,Loimaan seutu,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Pöytyän kunta,ei tietoa ylläpitäjästä,NaN,ei tietoa ylläpitoluokasta,NaN,2016/11/01,NaN,"LN, Luontokapinetti",251422.638706,6.757558e+06,2017/04/07,21900.0,636,25,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,34.0,58.0,34,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Särkijärven päivälaavu ja tulipaikka,NaN,NaN,Särkijärventie 21,Pöytyä,Loimaan seutu,Varsinais-Suomi,Kuhankuonon retkeilyreitistöön kuuluva kohde.,NaN,NaN,NaN,NaN,"puucee, laavu ja nuotiopaikka",www.kuhankuono.fi,NaN,NaN,NaN,NaN,Yhdistys,Leijankorven yhteismetsä ry,Kuhankuonon retkeilyreitistö- ja virkistysalue...,http://www.kuhankuono.fi/fi/palaute/,jatkuva,NaN,NaN,NaN,LN,246257.773768,6.747123e+06,2017/04/04,21900.0,636,25,2,NaN,F,T,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,465.0,491.0,465,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Retki- tai luonnonsatama,Utfärds- eller naturhamn,Excursion or natural harbour,Höglandin kiinnityslenkki,NaN,NaN,osoitteeton,Kemiönsaari,Turunmaa,Varsinais-Suomi,Sijaitsee Saaristomeren kansallispuistossa Hög...,NaN,NaN,NaN,NaN,NaN,http://www.luontoon.fi/saaristomeri/palvelut/h...,http://www.utinaturen.fi/sv/skargardshavet/ser...,http://www.nationalparks.fi/en/archipelagonp/s...,+358401234567,erkki@esimerkki.fi,Valtio,Metsähallitus,Metsähallitus,"Alueetta hoitaa Metsähallitus, osoite: Ratatie...",ei tietoa ylläpitoluokasta,NaN,1899/12/30,NaN,sannamari,239521.333937,

ainoa kysymysmerkki enään: <br>
- hakkenpään uimaranta <br>
    - oli liian kaukana toisistaan, niin haku ei bongannu sitä onko edes samat pisteet
    - täytyykö lisätä manuaalisesti
    - poistetaanko virma duplicaatit


In [167]:
# duplikaattienpoisto, jos nii ja koordinaatit mätsäävät

##not_matched = not_matched.drop_duplicates(subset=["name_fi", "x_eureffin", "y_eureffin"], keep="first")

In [168]:
not_matched[not_matched["name_fi"] == "Hakkeenpään ranta"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
829,1095.0,1122.0,1095,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Hakkeenpään ranta,NaN,NaN,Hakkeenpääntie 812,Taivassalo,Vakka-Suomi,Varsinais-Suomi,Hakkeenpään uimapaikassa on loiva hiekkaranta....,NaN,NaN,NaN,"Hiekkarantaan on esteetön pääsy, pukukoppeihin...","Käymälä, pukukopit, pöytiä ja penkkejä, nuotio...",NaN,NaN,NaN,NaN,NaN,Yhdistys,Hakkenpään kyläyhdistys,NaN,NaN,jatkuva,Hyvä,2018/08/16,NaN,"TA, TerhiSalosta, SaloMK",204285.537296,6.719082e+06,2020/12/04,23310.0,833,24,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [169]:
true_matches.shape

(276, 62)

In [170]:
true_matches = pd.concat([true_matches, not_matched[(not_matched["name_fi"] == "Hakkeenpään ranta")]], ignore_index=True)
true_matches.shape

(277, 62)

In [171]:
not_matched.shape

(1043, 62)

In [172]:
not_matched = not_matched[~not_matched["gid"].isin(true_matches["gid"].unique())]
not_matched.shape

(1042, 62)

In [173]:
true_matches.loc[true_matches["gid"] == 1095, ["id_2", "nimi_fi", "tyyppi_nimi_fi"]] = [510287, "Hakkenpään uimapaikka", "Uimapaikka"]
true_matches[true_matches["name_fi"] == "Hakkeenpään ranta"]

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
276,1095.0,1122.0,1095,Virkistyskohde,rekreationsobjekt,recreational attraction,Uimapaikka,badplats,unattended beach,Hakkeenpään ranta,NaN,NaN,Hakkeenpääntie 812,Taivassalo,Vakka-Suomi,Varsinais-Suomi,Hakkeenpään uimapaikassa on loiva hiekkaranta....,NaN,NaN,NaN,"Hiekkarantaan on esteetön pääsy, pukukoppeihin...","Käymälä, pukukopit, pöytiä ja penkkejä, nuotio...",NaN,NaN,NaN,NaN,NaN,Yhdistys,Hakkenpään kyläyhdistys,NaN,NaN,jatkuva,Hyvä,2018/08/16,NaN,"TA, TerhiSalosta, SaloMK",204285.537296,6.719082e+06,2020/12/04,23310.0,833,24,2,NaN,F,F,NaN,NaN,False,notknown,NaN,510287.0,Uimapaikka,Hakkenpään uimapaikka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [174]:
true_matches.loc[true_matches["class2_fi"].isin(["Uimapaikka", "Uimaranta"]),
    ["class2_fi", "tyyppi_nimi_fi", "name_fi", "nimi_fi"]]


,class2_fi,tyyppi_nimi_fi,name_fi,nimi_fi
0,Uimapaikka,Uimapaikka,Poikon uimapaikka,Poikon uimapaikka
2,Uimaranta,Uimaranta,Ekvallan uimaranta,Ekvallan uimaranta
3,Uimapaikka,Uimapaikka,Kalikan uimaranta,Pöytyän Kalikan uimaranta
5,Uimapaikka,Uimaranta,Mellilänjärven uimaranta,Mellilänjärven uimaranta
6,Uimapaikka,Uimapaikka,Pappistenjärven uimapaikka,Pappisten uimapaikka
...,...,...,...,...
258,Uimapaikka,Uimapaikka,Pitkäjärven uimaranta,Pitkäjärven uimaranta
259,Uimaranta,Uimapaikka,Ölmosin uimaranta,Västerbuktenin uimaranta
260,Uimaranta,Uimapaikka,Bläsnäs,Bläsnäsin uimala
263,Uimapaikka,Telttailu ja leiriytyminen,Auran kunnan Koivuniemen virkistysalue,Koivuniemen virkistysalue


In [192]:
not_matched.shape

(1042, 62)

In [175]:
virma_lipas_mapping_df =  pd.concat([not_matched, true_matches], axis=0, ignore_index=True)
virma_lipas_mapping_df

,id,fid,gid,class1_fi,class1_se,class1_en,class2_fi,class2_se,class2_en,name_fi,name_se,name_en,address,municipali,subregion,region,info_fi,info_se,info_en,chall_clas,accessibil,equipment,www_fi,www_se,www_en,telephone,email,ownerclass,owner,upkeeper,upkeepinfo,upkeepclas,shapeestim,sh_es_date,sh_es_pers,updater_id,x_eureffin,y_eureffin,timestamp,zip,munici_nro,subreg_nro,region_nro,special,no_address,publicinfo,picture,www_picture,hidden,datasource,lipas_id,id_2,tyyppi_nimi_fi,nimi_fi,n,distance,feature_x,feature_y,nearest_x,nearest_y,mapped_lipas_category,category_match
0,29.0,53.0,29,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Norrbynrannan uimapaikka,NaN,NaN,Tuulenpuoli 5,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,241444.642157,6.696252e+06,2017/04/05,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30.0,54.0,30,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Vepon uimapaikka,NaN,NaN,Veponranta 17,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yritys,NordKalk Oyj,ei ylläpitoa,ei ylläpitoa,ei ylläpitoa,NaN,NaN,NaN,LN,239880.642157,6.696266e+06,2017/04/04,21600.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31.0,55.0,31,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Uimapaikka,Badplats,Unattended beach,Simonbyn uimaranta,NaN,NaN,Harjulantie 120,Parainen,Turunmaa,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Paraisten kaupunki,ei ylläpitoa,NaN,ei ylläpitoa,NaN,2016/11/01,NaN,LN,225128.077680,6.684804e+06,2017/04/06,21650.0,573,21,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,33.0,57.0,33,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Luonto- tai lintutorni,Natur- eller fågeltorn,Nature or bird watching tower,Luontokapinetin näkötorni,NaN,NaN,Hovilanmäentie 2,Pöytyä,Loimaan seutu,Varsinais-Suomi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kunta,Pöytyän kunta,ei tietoa ylläpitäjästä,NaN,ei tietoa ylläpitoluokasta,NaN,2016/11/01,NaN,"LN, Luontokapinetti",251422.638706,6.757558e+06,2017/04/07,21900.0,636,25,2,NaN,F,F,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,34.0,58.0,34,Virkistyskohde,Rekreationsobjekt,Recreational attraction,Päivälaavu tai -kota,Dagvindskydd eller -kåta,Lean-to or lap pole tent for day time use,Särkijärven päivälaavu ja tulipaikka,NaN,NaN,Särkijärventie 21,Pöytyä,Loimaan seutu,Varsinais-Suomi,Kuhankuonon retkeilyreitistöön kuuluva kohde.,NaN,NaN,NaN,NaN,"puucee, laavu ja nuotiopaikka",www.kuhankuono.fi,NaN,NaN,NaN,NaN,Yhdistys,Leijankorven yhteismetsä ry,Kuhankuonon retkeilyreitistö- ja virkistysalue...,http://www.kuhankuono.fi/fi/palaute/,jatkuva,NaN,NaN,NaN,LN,246257.773768,6.747123e+06,2017/04/04,21900.0,636,25,2,NaN,F,T,NaN,NaN,False,notknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314,NaN,NaN,302554,Virkistyskohde,rekreationsobjekt,recreational attraction,Melontareitin lähtöpiste,startpunkt för paddlingsleden,starting point for a paddling route,Stora Masugnsträsketin kanoottilaituri ja vuok...,Stora Masugnsträskets kanotbrygga och hyreskan...,Stora Masugnsträsket canoes and canoe pier,NaN,Kemiönsaari,Åboland-Turunmaa,Varsinais-Suomi,2 kpl intiaanikanootteja. Kanootteja varataan ...,Uthyrningen av två kanoter sker under sommaren...,You can rent two canoes for exploring the lake...,NaN,NaN,NaN,https://www.visitkimitoon.fi/fi/kanootti,NaN,NaN,NaN,NaN,Kunta,NaN,NaN,NaN,ajoittainen,NaN,2022/12/21,NaN,sannamari,249626.50000

In [176]:
virma_lipas_mapping_df = virma_lipas_mapping_df[["id", "gid", "id_2"]].copy()
virma_lipas_mapping_df

,id,gid,id_2
0,29.0,29,NaN
1,30.0,30,NaN
2,31.0,31,NaN
3,33.0,33,NaN
4,34.0,34,NaN
...,...,...,...
1314,NaN,302554,609978.0
1315,340.0,340,99591.0
1316,489.0,489,100201.0
1317,NaN,302291,613417.0


In [177]:
virma_lipas_mapping_df.rename(columns={"id_2": "mapped_lipas_id", "id": "virma_id"}, inplace=True)
virma_lipas_mapping_df

,virma_id,gid,mapped_lipas_id
0,29.0,29,NaN
1,30.0,30,NaN
2,31.0,31,NaN
3,33.0,33,NaN
4,34.0,34,NaN
...,...,...,...
1314,NaN,302554,609978.0
1315,340.0,340,99591.0
1316,489.0,489,100201.0
1317,NaN,302291,613417.0


In [178]:
virma_lipas_mapping_df[["virma_id", "mapped_lipas_id"]] = virma_lipas_mapping_df[["virma_id", "mapped_lipas_id"]].astype("Int64")

In [179]:
virma_lipas_mapping_df.dtypes

virma_id           Int64
gid                int64
mapped_lipas_id    Int64
dtype: object

In [180]:
virma_lipas_mapping_df["mapped_lipas_id"] = virma_lipas_mapping_df["mapped_lipas_id"].fillna(0)
virma_lipas_mapping_df

,virma_id,gid,mapped_lipas_id
0,29,29,0
1,30,30,0
2,31,31,0
3,33,33,0
4,34,34,0
...,...,...,...
1314,<NA>,302554,609978
1315,340,340,99591
1316,489,489,100201
1317,<NA>,302291,613417


In [181]:
virma_lipas_mapping_df.dtypes

virma_id           Int64
gid                int64
mapped_lipas_id    Int64
dtype: object

In [182]:
virma_lipas_mapping_df.to_csv("virma_lipas_points_real.csv", index=False)

In [183]:
dupes_all = not_matched[not_matched.duplicated(subset=["name_fi", "x_eureffin", "y_eureffin"], keep=False)]

In [184]:
virma_lipas_points_matched = true_matches[["id", "gid", "id_2", "name_fi", "nimi_fi"]].copy()
virma_lipas_points_matched

,id,gid,id_2,name_fi,nimi_fi
0,1062.0,1062,529752.0,Poikon uimapaikka,Poikon uimapaikka
1,1150.0,1150,611083.0,Järvelän näköala- ja lintutorni,Järvelän näköala- ja lintutorni
2,133.0,133,100824.0,Ekvallan uimaranta,Ekvallan uimaranta
3,36.0,36,514585.0,Kalikan uimaranta,Pöytyän Kalikan uimaranta
4,53.0,53,524388.0,Suojalan hiihtomaja,Suojalan hiihtomaja
...,...,...,...,...,...
272,NaN,302554,609978.0,Stora Masugnsträsketin kanoottilaituri ja vuok...,Stora Masugnsträsketin kanoottilaituri
273,340.0,340,99591.0,Rautilan rantasauna,Rautilan sauna ja uimapaikka
274,489.0,489,100201.0,Himoisten rantasauna,Himoisten sauna ja uimaranta
275,NaN,302291,613417.0,"Aseman monitoimikenttä, jääkaukalo",Aseman kentän monitoimikaukalo


In [185]:
virma_lipas_points_matched.rename(columns={"id_2": "mapped_lipas_id", "id": "virma_id", "name_fi": "name_virma", "nimi_fi": "name_lipas"}, inplace=True)
virma_lipas_points_matched[["virma_id", "mapped_lipas_id"]] = virma_lipas_points_matched[["virma_id", "mapped_lipas_id"]].astype("Int64")
virma_lipas_points_matched

,virma_id,gid,mapped_lipas_id,name_virma,name_lipas
0,1062,1062,529752,Poikon uimapaikka,Poikon uimapaikka
1,1150,1150,611083,Järvelän näköala- ja lintutorni,Järvelän näköala- ja lintutorni
2,133,133,100824,Ekvallan uimaranta,Ekvallan uimaranta
3,36,36,514585,Kalikan uimaranta,Pöytyän Kalikan uimaranta
4,53,53,524388,Suojalan hiihtomaja,Suojalan hiihtomaja
...,...,...,...,...,...
272,<NA>,302554,609978,Stora Masugnsträsketin kanoottilaituri ja vuok...,Stora Masugnsträsketin kanoottilaituri
273,340,340,99591,Rautilan rantasauna,Rautilan sauna ja uimapaikka
274,489,489,100201,Himoisten rantasauna,Himoisten sauna ja uimaranta
275,<NA>,302291,613417,"Aseman monitoimikenttä, jääkaukalo",Aseman kentän monitoimikaukalo


In [186]:
virma_lipas_points_matched.dtypes

virma_id            Int64
gid                 int64
mapped_lipas_id     Int64
name_virma         object
name_lipas         object
dtype: object

In [187]:
virma_lipas_points_matched["lipas_link"] = virma_lipas_points_matched["mapped_lipas_id"].apply(
    lambda x: f"https://www.lipas.fi/liikuntapaikat/{x}" if pd.notna(x) else None
)
virma_lipas_points_matched

,virma_id,gid,mapped_lipas_id,name_virma,name_lipas,lipas_link
0,1062,1062,529752,Poikon uimapaikka,Poikon uimapaikka,https://www.lipas.fi/liikuntapaikat/529752
1,1150,1150,611083,Järvelän näköala- ja lintutorni,Järvelän näköala- ja lintutorni,https://www.lipas.fi/liikuntapaikat/611083
2,133,133,100824,Ekvallan uimaranta,Ekvallan uimaranta,https://www.lipas.fi/liikuntapaikat/100824
3,36,36,514585,Kalikan uimaranta,Pöytyän Kalikan uimaranta,https://www.lipas.fi/liikuntapaikat/514585
4,53,53,524388,Suojalan hiihtomaja,Suojalan hiihtomaja,https://www.lipas.fi/liikuntapaikat/524388
...,...,...,...,...,...,...
272,<NA>,302554,609978,Stora Masugnsträsketin kanoottilaituri ja vuok...,Stora Masugnsträsketin kanoottilaituri,https://www.lipas.fi/liikuntapaikat/609978
273,340,340,99591,Rautilan rantasauna,Rautilan sauna ja uimapaikka,https://www.lipas.fi/liikuntapaikat/99591
274,489,489,100201,Himoisten rantasauna,Himoisten sauna ja uimaranta,https://www.lipas.fi/liikuntapaikat/100201
275,<NA>,302291,613417,"Aseman monitoimikenttä, jääkaukalo",Aseman kentän monitoimikaukalo,https://www.lipas.fi/liikuntapaikat/613417


In [188]:
virma_lipas_points_matched.to_csv("virma_lipas_points_matched.csv", index=False, na_rep="NULL")